In [1]:
!nvidia-smi

Sat Nov 21 12:20:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q sacrebleu transformers accelerate

In [4]:
import ast, copy, gzip, io, json, os, random, re, statistics, subprocess, sys, tempfile, tokenize, urllib.request
from concurrent.futures import ThreadPoolExecutor
from math import comb

import sacrebleu

HUMANEVAL_URL = "https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz"


def load_humaneval():
    raw = gzip.decompress(urllib.request.urlopen(HUMANEVAL_URL).read()).decode()
    return [json.loads(l) for l in raw.splitlines()]


def pass_at_k(n, c, k):
    """Unbiased estimator: 1 - C(n-c, k) / C(n, k)."""
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)


INSTRUCTION = ("Complete the following Python function. Return the complete function in a single "
               "```python code block and nothing else.\n\n")
STOP_SEQS = ["\ndef ", "\nclass ", "\nif __name__", "\nprint(", "\n#"]


def extract_code(text, mode="instruct"):
    if mode == "base":
        for s in STOP_SEQS:
            text = text.split(s)[0]
        return text
    blocks = re.findall(r"```(?:python|py)?\n(.*?)```", text, re.S)
    if blocks:
        return blocks[0]
    if "```" in text:
        return text.split("```", 1)[1].split("\n", 1)[-1]
    return text


def build_program(p, completion, mode="instruct"):
    code = extract_code(completion, mode)
    if mode == "base":
        return p["prompt"] + code
    if f"def {p['entry_point']}" in code:
        return p["prompt"] + "    pass\n\n" + code
    return p["prompt"] + code


def run_program(src, timeout=10):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(src)
        path = f.name
    try:
        r = subprocess.run([sys.executable, "-I", path], capture_output=True, timeout=timeout)
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)


def is_correct(p, program):
    return run_program(program + "\n\n" + p["test"] + f"\n\ncheck({p['entry_point']})\n")


def _entry_source(src, entry):
    """Source of the (last) entry function without docstring, normalised by ast.unparse; falls back to raw text."""
    try:
        tree = ast.parse(src)
        fns = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == entry]
        fn = copy.deepcopy(fns[-1])
        if fn.body and isinstance(fn.body[0], ast.Expr) and isinstance(getattr(fn.body[0], "value", None), ast.Constant) \
                and isinstance(fn.body[0].value.value, str):
            fn.body = fn.body[1:] or [ast.Pass()]
        return ast.unparse(fn)
    except Exception:
        return src


def _tokens(src):
    try:
        return [t.string for t in tokenize.generate_tokens(io.StringIO(src).readline)
                if t.type not in (tokenize.NEWLINE, tokenize.NL, tokenize.INDENT, tokenize.DEDENT,
                                  tokenize.ENDMARKER, tokenize.COMMENT)]
    except Exception:
        return src.split()


def bleu_vs_reference(p, program):
    ref = _entry_source(p["prompt"] + p["canonical_solution"], p["entry_point"])
    cand = _entry_source(program, p["entry_point"])
    return sacrebleu.sentence_bleu(" ".join(_tokens(cand)), [" ".join(_tokens(ref))], tokenize="none").score


def auc(pos, neg):
    if not pos or not neg:
        return float("nan")
    return sum((a > b) + 0.5 * (a == b) for a in pos for b in neg) / (len(pos) * len(neg))


def evaluate(problems, generate, n, temperature, mode="instruct", workers=2):
    """generate(prompt, n, temperature) -> list of n raw model outputs. Returns per-problem records."""
    records = []
    for i, p in enumerate(problems):
        prompt = (INSTRUCTION if mode == "instruct" else "") + p["prompt"]
        outs = generate(prompt, n, temperature)
        programs = [build_program(p, o, mode) for o in outs]
        with ThreadPoolExecutor(workers) as ex:
            ok = list(ex.map(lambda s: is_correct(p, s), programs))
        bl = [bleu_vs_reference(p, s) for s in programs]
        records.append(dict(task_id=p["task_id"], n=n, c=sum(ok), correct=ok, bleu=bl, outputs=outs))
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(problems)} problems done", flush=True)
    return records


def summarize(records, ks=(1,)):
    res = {}
    for k in ks:
        ks_ok = [r for r in records if r["n"] >= k]
        res[f"pass@{k}"] = 100 * statistics.mean(pass_at_k(r["n"], r["c"], k) for r in ks_ok)
    flat = [(b, c) for r in records for b, c in zip(r["bleu"], r["correct"])]
    good = [b for b, c in flat if c]
    bad = [b for b, c in flat if not c]
    res["n_samples"] = len(flat)
    res["n_correct"] = len(good)
    res["bleu_mean_correct"] = statistics.mean(good) if good else float("nan")
    res["bleu_mean_incorrect"] = statistics.mean(bad) if bad else float("nan")
    res["auc_bleu_predicts_correct"] = auc(good, bad)
    return res

In [5]:
_probs = load_humaneval()
print(len(_probs), "bài")
def _fake(prompt, n, temperature):
    p = next(q for q in _probs if q["prompt"] in prompt)
    return ["```python\n" + p["prompt"] + p["canonical_solution"] + "```"] * n
_rec = evaluate(_probs[:10], _fake, n=2, temperature=0.0)
print("pass@1 (kỳ vọng 100.0):", summarize(_rec, ks=(1,))["pass@1"])

164 bài
  10/10 problems done
pass@1 (kỳ vọng 100.0): 100.0


In [6]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MODE = "instruct"
MAX_PROBLEMS = 20
N_SAMPLES = 10
TEMPERATURE = 0.2
MAX_NEW_TOKENS = 512

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Đã tải xong:", MODEL_NAME, "| thiết bị:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã tải xong: Qwen/Qwen2.5-Coder-1.5B-Instruct | thiết bị: cuda:0


In [8]:
def generate(prompt, n, temperature):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) if MODE == "instruct" else prompt
    ids = tok(text, return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=0.95, num_return_sequences=n)
    else:
        kw.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**ids, **kw)
    outs = [tok.decode(o[ids["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]
    return outs * n if temperature == 0 else outs

In [9]:
probs = load_humaneval()
if MAX_PROBLEMS: probs = probs[:MAX_PROBLEMS]
print(generate(INSTRUCTION + probs[0]["prompt"], 1, 0.0)[0])

```python
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
```


In [10]:
rec_greedy = evaluate(probs, generate, n=1, temperature=0.0, mode=MODE)
res_greedy = summarize(rec_greedy, ks=(1,))
print(res_greedy)

  10/20 problems done
  20/20 problems done
{'pass@1': 80.0, 'n_samples': 20, 'n_correct': 16, 'bleu_mean_correct': 58.79914203586245, 'bleu_mean_incorrect': 38.395312903150334, 'auc_bleu_predicts_correct': 0.765625}


In [11]:
rec_samp = evaluate(probs, generate, n=N_SAMPLES, temperature=TEMPERATURE, mode=MODE)
res_samp = summarize(rec_samp, ks=(1, 10))
print(res_samp)

  10/20 problems done
  20/20 problems done
{'pass@1': 82.5, 'pass@10': 95.0, 'n_samples': 200, 'n_correct': 165, 'bleu_mean_correct': 59.47464530943105, 'bleu_mean_incorrect': 34.914278583538554, 'auc_bleu_predicts_correct': 0.8600865800865801}


In [12]:
print("=== KẾT QUẢ TN4 ===")
print("Mô hình:", MODEL_NAME, "| số bài:", len(probs))
print(f"Greedy   pass@1  = {res_greedy['pass@1']:.1f}%")
print(f"Sampling pass@1  = {res_samp['pass@1']:.1f}%   pass@10 = {res_samp['pass@10']:.1f}%")
print(f"BLEU trung bình mẫu đúng   = {res_samp['bleu_mean_correct']:.1f}")
print(f"BLEU trung bình mẫu sai    = {res_samp['bleu_mean_incorrect']:.1f}")
print(f"AUC (BLEU dự đoán đúng/sai) = {res_samp['auc_bleu_predicts_correct']:.3f}")
json.dump(dict(model=MODEL_NAME, n_problems=len(probs), greedy=res_greedy, sampling=res_samp,
               rec_greedy=rec_greedy, rec_samp=rec_samp), open("tn4_results.json", "w"))
print("Đã lưu tn4_results.json (có thể tải về từ thanh Files bên trái).")

=== KẾT QUẢ TN4 ===
Mô hình: Qwen/Qwen2.5-Coder-1.5B-Instruct | số bài: 20
Greedy   pass@1  = 80.0%
Sampling pass@1  = 82.5%   pass@10 = 95.0%
BLEU trung bình mẫu đúng   = 59.5
BLEU trung bình mẫu sai    = 34.9
AUC (BLEU dự đoán đúng/sai) = 0.860
Đã lưu tn4_results.json (có thể tải về từ thanh Files bên trái).
